In [ ]:
# --- Kernel info (run this FIRST) ---------------------------------------------
# This lab's kernel works even on Python 3.10 (Azure ML SDK 1.61 imports AutoML
# fine here). This cell just reports the Python version — it no longer stops the
# notebook. Make sure you first ran install_requirements.ipynb and restarted the
# kernel so the SDK and a compatible ipywidgets (<8.0.0) are installed.
import sys

print("Python:", sys.version)
if sys.version_info[:2] != (3, 8):
    print("NOTE: not on Python 3.8. That's OK here as long as the SDK imports "
          "(verify with preflight_check.ipynb).")
else:
    print("Kernel OK — you are on Python 3.8.")


In [ ]:
from azureml.core import Workspace, Experiment

ws = Workspace.from_config()
exp = Experiment(workspace=ws, name="udacity-project")

print('Workspace name: ' + ws.name, 
      'Azure region: ' + ws.location, 
      'Subscription id: ' + ws.subscription_id, 
      'Resource group: ' + ws.resource_group, sep = '\n')

run = exp.start_logging()

In [ ]:
from azureml.core.compute import ComputeTarget, AmlCompute
from azureml.core.compute_target import ComputeTargetException

cluster_name = "project-cluster"

# Create compute cluster (reuse it if it already exists).
# Use vm_size = "Standard_D2_V2" and max_nodes no greater than 4.
try:
    compute_target = ComputeTarget(workspace=ws, name=cluster_name)
    print("Found existing compute target.")
except ComputeTargetException:
    print("Creating a new compute target...")
    compute_config = AmlCompute.provisioning_configuration(
        vm_size="Standard_D2_V2",
        max_nodes=4,
    )
    compute_target = ComputeTarget.create(ws, cluster_name, compute_config)

# Wait until the cluster is ready to accept jobs.
compute_target.wait_for_completion(show_output=True)
print("Compute target ready:", compute_target.name)


In [ ]:
from azureml.widgets import RunDetails
from azureml.train.hyperdrive.run import PrimaryMetricGoal
from azureml.train.hyperdrive.policy import BanditPolicy
from azureml.train.hyperdrive.sampling import RandomParameterSampling
from azureml.train.hyperdrive.runconfig import HyperDriveConfig
from azureml.train.hyperdrive.parameter_expressions import choice, uniform
from azureml.core import Environment, ScriptRunConfig
import os
import shutil

# Specify parameter sampler
ps = RandomParameterSampling({
    "--C": uniform(0.08, 0.1),                 # Inverse regularization strength
    "--max_iter": choice(25, 50, 100, 200),    # Maximum iterations
})

# Specify a Policy
policy = BanditPolicy(evaluation_interval=2, slack_factor=0.1, delay_evaluation=5)

# Make the training script available inside the source directory.
os.makedirs("./training", exist_ok=True)
shutil.copy("train.py", "./training")

# Build the training environment from conda_dependencies.yml. This is
# self-contained and does not rely on any curated environment being registered
# in the workspace (this workspace has none). train.py reads the public CSV with
# pandas, so scikit-learn + pandas + azureml-defaults is all the cluster needs.
sklearn_env = Environment.from_conda_specification(
    name="sklearn-env", file_path="conda_dependencies.yml"
)

# Create a ScriptRunConfig Object to specify the configuration details of your training job
src = ScriptRunConfig(
    source_directory="./training",
    script="train.py",
    compute_target=compute_target,
    environment=sklearn_env,
)

# Create a HyperDriveConfig using the src object, hyperparameter sampler, and policy.
hyperdrive_config = HyperDriveConfig(
    run_config=src,
    hyperparameter_sampling=ps,
    policy=policy,
    primary_metric_name="Accuracy",
    primary_metric_goal=PrimaryMetricGoal.MAXIMIZE,
    max_total_runs=20,
    max_concurrent_runs=4,
)


In [ ]:
# Submit your hyperdrive run to the experiment and show run details with the widget.
hyperdrive_run = exp.submit(hyperdrive_config)
RunDetails(hyperdrive_run).show()
hyperdrive_run.wait_for_completion(show_output=True)


In [ ]:
import joblib
# Get your best run and save the model from that run.

best_run = hyperdrive_run.get_best_run_by_primary_metric()
print(best_run)

best_run_metrics = best_run.get_metrics()
print(best_run_metrics)

In [ ]:
print(best_run.get_file_names())

In [ ]:
# Register the model
model = best_run.register_model(model_path='outputs/hyperdrive_model.joblib', model_name='hyperdrive_model',
                   tags={'Training context':'Parameterized SKLearn Estimator'})
model

In [ ]:
from azureml.data.dataset_factory import TabularDatasetFactory

# Create TabularDataset using TabularDatasetFactory
# Data is available at: 
# "https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv"

### YOUR CODE HERE ###
data_url = 'https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/bankmarketing_train.csv'
ds = TabularDatasetFactory.from_delimited_files(path=data_url)


In [ ]:
from train import clean_data
from sklearn.model_selection import train_test_split

x, y = clean_data(ds)

# Split data into train and test sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)
train_data = x_train.join(y_train)

In [ ]:
from azureml.train.automl import AutoMLConfig

# Set parameters for AutoMLConfig
# NOTE: DO NOT CHANGE THE experiment_timeout_minutes PARAMETER OR YOUR INSTANCE WILL TIME OUT.
# If you wish to run the experiment longer, you will need to run this notebook in your own
# Azure tenant, which will incur personal costs.
#
# ignore_package_version_incompatibilities=True skips AutoML's local package
# check, which otherwise fails because this environment has ipywidgets 8.x while
# azureml-widgets pins ipywidgets<8.0.0. The check is cosmetic (RunDetails
# widget) and does not affect training.
automl_config = AutoMLConfig(
    experiment_timeout_minutes=30,
    task='classification',
    debug_log='automl_errors.log',
    training_data=train_data,
    label_column_name='y',
    n_cross_validations=5,
    primary_metric='accuracy',
    ignore_package_version_incompatibilities=True,
)


In [ ]:
# Submit your automl run
automl_run = exp.submit(automl_config, show_output=True)


In [ ]:
# Retrieve and save your best automl model.
best_run, automl_best_model = automl_run.get_output()
joblib.dump(value = automl_best_model, filename = './outputs/automl_best_model.joblib')
print(best_run)
print(automl_best_model)

In [ ]:
from pprint import pprint
def print_model(model, prefix=""):
    for step in model.steps:
        print(prefix + step[0])
        if hasattr(step[1], 'estimators') and hasattr(step[1], 'weights'):
            pprint({'estimators': list(
                e[0] for e in step[1].estimators), 'weights': step[1].weights})
            print()
            for estimator in step[1].estimators:
                print_model(estimator[1], estimator[0] + ' - ')
        else:
            pprint(step[1].get_params())
            print()

print_model(automl_best_model)

In [ ]:
best_run.get_tags()

In [ ]:
compute_target.delete()